In [10]:
import numpy as np
import torch
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForCausalLM

### Scenerio generation

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=""
)

messages = [
    {
        "role": "system",
        "content": "You are a professional workplace scenario generator."
    },
    {
        "role": "user",
        "content": (
                    f"Create a short professional workplace scenario for a 25-year-old male "
                    f"working as a software developer. "
                    "The scenario must:\n"
                    "- Start with 'You are'\n"
                    "- Describe a challenging situation\n"
                    "- Involve pressure, conflict, or emotional stress\n"
                    "- Contain NO instructions, NO solutions, NO dialogue\n"
                    "- Be 3–4 sentences maximum"
                )
    }
]

response = client.chat.completions.create(
    messages=messages,
    max_tokens=120,
    temperature=0.6,
    top_p=0.9
)

print(response.choices[0].message.content)


You are Alex, a 25-year-old software developer at a mid-sized tech firm. The company's flagship project, a cutting-edge mobile app, is nearing its launch deadline, but the lead developer, Rachel, has just left the company unexpectedly, leaving you with a critical section of code that you're not familiar with. The project manager, John, is breathing down your neck to complete the task, and the marketing team is threatening to push back the launch date if the app isn't ready on time. The weight of responsibility is crushing, and you're starting to feel overwhelmed.


In [ ]:
class LocalEQScenarioGenerator:
    def __init__(self):
        print("Loading local EQ scenario model...")

        self.model_id = "EssentialAI/rnj-1-instruct"

        self.generator = pipeline(
            "text-generation",
            model=self.model_id,
            dtype=0 if torch.cuda.is_available() else -1,
            # device_map="auto"
        )

        print("Scenario generator ready.")

    def generate_scenario(self, user_profile: dict) -> str:
        """
        Generates ONLY a realistic workplace scenario.
        No advice. No actions. No questions.
        """

        age = user_profile.get("age")
        gender = user_profile.get("gender")
        profession = user_profile.get("profession")

        messages = [
            {"role": "user", "content": (f"Write a short professional workplace scenario.\n\n"
            f"Profile:\n"
            f"- Age: {age}\n"
            f"- Gender: {gender}\n"
            f"- Profession: {profession}\n\n"
            "Rules:\n"
            "- Start with 'You are'\n"
            "- Describe only the situation\n"
            "- Include pressure, conflict, or emotional stress\n"
            "- Do NOT give advice\n"
            "- Do NOT ask questions\n"
            "- Do NOT include dialogue\n"
            "- Maximum 4 sentences\n\n"
            "Scenario:\n")},
        ]

        output = self.generator(
            prompt,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.15,
            return_full_text=False,  # 🔥 VERY IMPORTANT
        )

        scenario = output[0]["generated_text"].strip()

        return scenario

scenario_generator = LocalEQScenarioGenerator()


Loading local EQ scenario model...


c:\Users\sange\Desktop\emotional_int_assessment\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sange\.cache\huggingface\hub\models--EssentialAI--rnj-1-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'at

In [ ]:
scenario_generator = LocalEQScenarioGenerator()

user_profile = {
    "age": 25,
    "gender": "Male",
    "profession": "Customer Service Agent"
}

scenario = scenario_generator.generate_scenario(user_profile)
print("\nSCENARIO:\n")
print(scenario)


In [20]:
class LocalEQScenarioGenerator:
    def __init__(self):
        print("Loading local EQ scenario model...")

        self.model_id = "Qwen/Qwen2.5-1.5B-Instruct"

        self.generator = pipeline(
            "text-generation",
            model=self.model_id,
            dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            # device_map="auto"
        )

        print("Scenario generator ready.")

    def generate_scenario(self, user_profile: dict) -> str:
        """
        Generates ONLY a realistic workplace scenario.
        No advice. No actions. No questions.
        """

        age = user_profile.get("age")
        gender = user_profile.get("gender")
        profession = user_profile.get("profession")

        messages = [
            {
                "role": "system",
                "content": (
                    "You generate realistic workplace scenarios for emotional intelligence assessments. "
                    "Describe ONLY the situation. Do NOT give advice, actions, or questions."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Create a short professional workplace scenario for a {age}-year-old {gender} "
                    f"working as a {profession}. "
                    "The scenario must:\n"
                    "- Start with 'You are'\n"
                    "- Describe a challenging situation\n"
                    "- Involve pressure, conflict, or emotional stress\n"
                    "- Contain NO instructions, NO solutions, NO dialogue\n"
                    "- Be 3–4 sentences maximum"
                )
            }
        ]

        prompt = self.generator.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        output = self.generator(
            prompt,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.2,
            truncation=True
        )

        text = output[0]["generated_text"]
        scenario = text.split("<|assistant|>")[-1].strip()

        return scenario

scenario_generator = LocalEQScenarioGenerator()


Loading local EQ scenario model...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


Scenario generator ready.


In [21]:
user_profile = {
    "age": 25,
    "gender": "Male",
    "profession": "Customer Service Agent"
}

scenario = scenario_generator.generate_scenario(user_profile)

print("\nFINAL SCENARIO:\n")
print(scenario)


KeyboardInterrupt: 

In [37]:
import torch
from transformers import pipeline


class LocalEQScenarioGenerator:
    def __init__(self):
        print("Loading EQ scenario model...")

        self.model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

        self.generator = pipeline(
            "text-generation",
            model=self.model_id,
            device=0 if torch.cuda.is_available() else -1
        )

        print("Model loaded successfully.")

    def generate_scenario(self, user_profile: dict) -> str:
        age = user_profile["age"]
        gender = user_profile["gender"]
        profession = user_profile["profession"]

        prompt = f"""
            You are generating OUTPUT TEXT, not instructions.

            RULES (MUST FOLLOW):
            - Output ONLY a workplace scenario
            - Do NOT explain how to write
            - Do NOT give advice
            - Do NOT give instructions
            - Do NOT ask questions
            - Do NOT describe actions to take
            - Start the text exactly with: "You are"
            - Maximum 4 sentences
            - Plain narrative text only

            PROFILE:
            Age: {age}
            Gender: {gender}
            Profession: {profession}

            OUTPUT:
            """

        output = self.generator(
            prompt,
            max_new_tokens=120,
            temperature=0.6,
            do_sample=True
        )

        text = output[0]["generated_text"]

        # Remove prompt echo
        scenario = text[len(prompt):].strip()

        return scenario


In [38]:
scenario_generator = LocalEQScenarioGenerator()

user_profile = {
    "age": 25,
    "gender": "Male",
    "profession": "Customer Service Agent"
}

scenario = scenario_generator.generate_scenario(user_profile)
print("\nSCENARIO:\n", scenario)


Loading EQ scenario model...


Device set to use cpu


Model loaded successfully.

SCENARIO:
 You are a customer service agent at a retail store. The store is open from 9:00 AM to 5:00 PM. The store has a policy of 20% off on all products during the first hour of the day.
            
            You are greeting customers, answering their questions, and helping them find the products they need.
            
            You are also responsible for tracking sales and ensuring that sales targets are achieved.
            
            The store has a team of 5 customer service agents who are responsible for ensuring that


In [14]:

# User Info
profession = "Customer Service Agent"
gender = "Male"
age = 25

# 1. Generate Scenario
# (Note: FLAN-T5 is brief. For longer stories, you might need 'gpt2-medium')
scenario_text = sceneriogenerator.generate_scenario_local(profession, gender, age)

print(f"\nGENERATED SCENARIO:\n{scenario_text}")



--- Generating Scenario for 25, Male and Customer Service Agent... ---


ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

### Questions generator

In [ ]:
from transformers import pipeline, set_seed
import torch
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. SETUP HUGGING FACE MODELS (LOCAL)
# ==========================================
class LocalEQSystem:
    def __init__(self):
        print("Loading models... (This uses your local RAM/GPU)")
        
        # MODEL 1: Emotion Detection
        # We use a specialized BERT model for emotion
        self.emotion_classifier = pipeline(
            "text-classification", 
            model="j-hartmann/emotion-english-distilroberta-base", 
            return_all_scores=True
        )
        
        # MODEL 2: Scenario Generation
        # We use FLAN-T5-Base because it follows instructions better than GPT-2
        # and runs fast on Google Colab / Local CPUs.
        self.generator = pipeline(
            "text2text-generation", 
            model="google/flan-t5-base", 
            max_length=512
        )
        print("Systems Ready.")

    def generate_scenario_local(self, profession, age):
        """
        Generates a scenario using local Hugging Face model.
        Since local models struggle with strict JSON, we use a string template.
        """
        print(f"\n--- Generating Scenario for {age}yo {profession}... ---")
        
        # Prompt Engineering for FLAN-T5
        prompt = (
            f"Write a stressful workplace scenario for a {age} year old {profession}. "
            "Then ask two difficult questions about how to handle it. "
            "Scenario:"
        )
        
        # Generate text
        output = self.generator(prompt, do_sample=True, temperature=0.8)
        generated_text = output[0]['generated_text']
        
        return generated_text

    def analyze_emotion(self, text):
        results = self.emotion_classifier(text)
        # Flatten structure: [{'label': 'joy', 'score': 0.9}, ...] -> {'joy': 0.9, ...}
        return {item['label']: item['score'] for item in results[0]}

    def calculate_score(self, emotions):
        # High EQ = High Joy/Neutral + Low Anger/Fear
        score = 50 
        score += (emotions.get('joy', 0) * 40)
        score += (emotions.get('neutral', 0) * 30)
        score -= (emotions.get('anger', 0) * 30)
        score -= (emotions.get('fear', 0) * 30)
        return max(0, min(100, score))

    def visualize(self, emotions, score):
        # Clean data for chart (remove tiny values)
        data = {k: v for k, v in emotions.items() if v > 0.02}
        
        plt.figure(figsize=(10, 4))
        
        # Pie Chart
        plt.subplot(1, 2, 1)
        plt.pie(data.values(), labels=data.keys(), autopct='%1.1f%%')
        plt.title("Emotional Analysis")
        
        # Score Bar
        plt.subplot(1, 2, 2)
        color = 'green' if score > 70 else 'orange' if score > 40 else 'red'
        plt.bar(["EQ Score"], [score], color=color)
        plt.ylim(0, 100)
        plt.title(f"Score: {int(score)}/100")
        
        plt.show()

# ==========================================
# 2. RUNNING THE SYSTEM
# ==========================================

# Initialize
system = LocalEQSystem()

# User Info
profession = "Customer Service Agent"
age = 25

# 1. Generate Scenario
# (Note: FLAN-T5 is brief. For longer stories, you might need 'gpt2-medium')
scenario_text = system.generate_scenario_local(profession, age)

print(f"\nGENERATED SCENARIO:\n{scenario_text}")
print("-" * 50)

# 2. Simulate User Response
print("Enter your response below:")
# user_input = input() # Uncomment to type manually
user_input = "I would remain calm, listen to the customer's problem without interrupting, and apologize for the inconvenience."
print(f"User: {user_input}")

# 3. Analyze
emotions = system.analyze_emotion(user_input)
final_score = system.calculate_score(emotions)

# 4. Show Result
system.visualize(emotions, final_score)

In [19]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

inputs = tokenizer("I would feel frustrated initially, but I understand people can make mistakes.", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
print(logits)
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]


tensor([[-0.8473,  0.8732]])


'POSITIVE'

In [5]:
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

emotion_analyzer = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)


Device set to use cpu
Device set to use cpu
c:\Users\sange\Desktop\emotional_int_assessment\.venv\lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [6]:
user_profile = {
    "age": 24,
    "gender": "Female",
    "profession": "Software Engineer"
}


In [7]:
scenario = (
    "You are working on a critical project with a tight deadline. "
    "A teammate misses an important task, affecting progress."
)

questions = [
    "How would you feel in this situation?",
    "How would you respond to your teammate?",
    "What steps would you take to manage your emotions?"
]

print("Scenario:\n", scenario)
print("\nQuestions:")
for q in questions:
    print("-", q)


Scenario:
 You are working on a critical project with a tight deadline. A teammate misses an important task, affecting progress.

Questions:
- How would you feel in this situation?
- How would you respond to your teammate?
- What steps would you take to manage your emotions?


In [8]:
responses = [
    "I would feel frustrated initially, but I understand people can make mistakes.",
    "I would talk calmly with them and try to find a solution together.",
    "I would take a moment to calm myself and focus on resolving the issue."
]


In [9]:
def validate_response(text, min_words=5):
    if len(text.split()) < min_words:
        return False
    return True

validated_responses = [r for r in responses if validate_response(r)]

validated_responses


['I would feel frustrated initially, but I understand people can make mistakes.',
 'I would talk calmly with them and try to find a solution together.',
 'I would take a moment to calm myself and focus on resolving the issue.']

In [21]:
def get_sentiment_scores(text):
    result = sentiment_analyzer(text)[0]
    return result["label"], result["score"]

sentiment_results = [get_sentiment_scores(r) for r in validated_responses]
sentiment_results


[('POSITIVE', 0.8481934070587158),
 ('NEGATIVE', 0.9950717091560364),
 ('NEGATIVE', 0.9970566034317017)]

In [11]:
def get_emotion_scores(text):
    emotions = emotion_analyzer(text)[0]
    return {e["label"]: e["score"] for e in emotions}

emotion_results = [get_emotion_scores(r) for r in validated_responses]
emotion_results


[{'anger': 0.987751841545105,
  'disgust': 0.00292025925591588,
  'fear': 0.0013783269096165895,
  'joy': 0.0004166270955465734,
  'neutral': 0.002982943784445524,
  'sadness': 0.003522442886605859,
  'surprise': 0.0010274965316057205},
 {'anger': 0.37216252088546753,
  'disgust': 0.06587148457765579,
  'fear': 0.043937575072050095,
  'joy': 0.08825377374887466,
  'neutral': 0.39161422848701477,
  'sadness': 0.03613114729523659,
  'surprise': 0.002029233844950795},
 {'anger': 0.02386077493429184,
  'disgust': 0.025029970332980156,
  'fear': 0.028145598247647285,
  'joy': 0.07496796548366547,
  'neutral': 0.6446672081947327,
  'sadness': 0.2000001072883606,
  'surprise': 0.0033283112570643425}]

In [12]:
EQ_CATEGORIES = {
    "self_awareness": ["joy", "sadness"],
    "emotional_resilience": ["fear", "sadness"],
    "conflict_resolution": ["anger"],
    "empathy": ["joy"],
    "emotional_regulation": ["anger", "fear"]
}


In [13]:
def calculate_eq_scores(emotion_results):
    scores = {cat: 0.0 for cat in EQ_CATEGORIES}

    for emotions in emotion_results:
        for category, relevant_emotions in EQ_CATEGORIES.items():
            for emo in relevant_emotions:
                scores[category] += emotions.get(emo, 0)

    # Normalize
    for k in scores:
        scores[k] = round(scores[k] / len(emotion_results), 3)

    return scores

eq_scores = calculate_eq_scores(emotion_results)
eq_scores


{'self_awareness': 0.134,
 'emotional_resilience': 0.104,
 'conflict_resolution': 0.461,
 'empathy': 0.055,
 'emotional_regulation': 0.486}

In [14]:
overall_eq = round(np.mean(list(eq_scores.values())) * 100, 2)
overall_eq


np.float64(24.8)

In [15]:
def interpret_eq(score):
    if score < 40:
        return "Low EQ – Needs emotional skill development"
    elif score < 70:
        return "Average EQ – Good emotional awareness"
    else:
        return "High EQ – Strong emotional intelligence"

interpretation = interpret_eq(overall_eq)
interpretation


'Low EQ – Needs emotional skill development'

In [16]:
print("User Profile:", user_profile)
print("Overall EQ Score:", overall_eq)
print("Interpretation:", interpretation)
print("\nCategory Scores:")
for k, v in eq_scores.items():
    print(f"{k}: {v}")


User Profile: {'age': 24, 'gender': 'Female', 'profession': 'Software Engineer'}
Overall EQ Score: 24.8
Interpretation: Low EQ – Needs emotional skill development

Category Scores:
self_awareness: 0.134
emotional_resilience: 0.104
conflict_resolution: 0.461
empathy: 0.055
emotional_regulation: 0.486
